# 05 — LIMU-BERT Self-Supervised IMU Pretraining

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Section 16:** Self-supervised IMU representation learning/pretraining via Masked Sensor Modeling (MSM).
> **Section 13:** Remote GPU execution requirement (`torch.cuda.is_available() == True`).
> **Section 58:** GPU training with mixed precision, AdamW optimizer, and validation checkpointing.

### Objectives:
1. Train LIMU-BERT on real IO-VNBD IMU sequence windows (6-DOF: 3-axis accel + 3-axis gyro).
2. Mask 15% of sensor timesteps and reconstruct true signals using multi-head self-attention.
3. Learn temporal correlations, filter residual noise, and build robust inertial embeddings.
4. Save best checkpoint to `checkpoints/limu_bert/limu_bert_best.pt` for downstream odometry networks.

## 1. Environment & GPU Verification (Section 13)

In [ ]:
import os, sys, json, time
from pathlib import Path
import yaml
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

# Strict GPU Preflight Check (Section 13)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Execution Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('[NOTICE] CUDA not detected on local host. Remote Lightning GPU execution required for production training.')

ckpt_dir = PROJECT_ROOT / 'checkpoints' / 'limu_bert'
plots_dir = PROJECT_ROOT / 'plots' / 'limu_bert'
ckpt_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Configuration & Datasets (Section 48)

In [ ]:
with open(PROJECT_ROOT / 'configs' / 'training.yaml', 'r') as f:
    cfg = yaml.safe_load(f)['limu_bert']

print('LIMU-BERT Configuration:')
print(json.dumps(cfg, indent=2))

from src.datasets.limu_bert_dataset import LIMUBERTDataset
from src.models.limu_bert import LIMUBERT

train_ds = LIMUBERTDataset(
    split='train',
    window_size=120,
    stride=40,
    mask_ratio=cfg.get('mask_ratio', 0.15)
)

val_ds = LIMUBERTDataset(
    split='val',
    window_size=120,
    stride=60,
    mask_ratio=cfg.get('mask_ratio', 0.15)
)

batch_size = cfg.get('batch_size', 64)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

print(f'Train Batches: {len(train_loader)} | Validation Batches: {len(val_loader)}')

## 3. Instantiate LIMU-BERT Model & Optimizer

In [ ]:
model = LIMUBERT(
    input_dim=6,
    hidden_dim=cfg.get('hidden_dim', 128),
    num_heads=cfg.get('num_heads', 4),
    num_layers=cfg.get('num_layers', 4),
    dropout=cfg.get('dropout', 0.1)
).to(device)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'LIMU-BERT Trainable Parameters: {param_count:,}')

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(cfg.get('learning_rate', 1.0e-3)),
    weight_decay=float(cfg.get('weight_decay', 1.0e-4))
)

num_epochs = cfg.get('num_epochs', 40)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

## 4. Self-Supervised Training Loop (Section 37 & 58)

In [ ]:
train_losses = []
val_losses = []
best_val_loss = float('inf')
best_ckpt_path = ckpt_dir / 'limu_bert_best.pt'

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
print(f'Starting LIMU-BERT training for {num_epochs} epochs...')

for epoch in range(1, num_epochs + 1):
    model.train()
    total_train_loss = 0.0
    n_train_batches = 0
    
    for batch in train_loader:
        x_in = batch['input'].to(device)
        target = batch['target'].to(device)
        mask = batch['mask'].to(device)
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            recon, _ = model(x_in)
            loss = model.compute_pretraining_loss(recon, target, mask)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_train_loss += loss.item()
        n_train_batches += 1
        
    scheduler.step()
    avg_train_loss = total_train_loss / max(1, n_train_batches)
    train_losses.append(avg_train_loss)
    
    # Validation Loop
    model.eval()
    total_val_loss = 0.0
    n_val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            x_in = batch['input'].to(device)
            target = batch['target'].to(device)
            mask = batch['mask'].to(device)
            
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                recon, _ = model(x_in)
                v_loss = model.compute_pretraining_loss(recon, target, mask)
            total_val_loss += v_loss.item()
            n_val_batches += 1
            
    avg_val_loss = total_val_loss / max(1, n_val_batches)
    val_losses.append(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
            'config': cfg,
            'param_count': param_count
        }, best_ckpt_path)
        star = ' * [CHECKPOINT SAVED]'
    else:
        star = ''
        
    if epoch % 5 == 0 or epoch == 1 or star:
        print(f'Epoch [{epoch:3d}/{num_epochs:3d}] | Train MSE: {avg_train_loss:.5f} | Val MSE: {avg_val_loss:.5f}{star}')

print(f'Training complete! Best Validation Loss: {best_val_loss:.5f}')
print(f'Saved checkpoint: {best_ckpt_path}')

## 5. Training Curves Visualization & Reconstruction Inspection

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, len(train_losses) + 1), train_losses, 'b-', label='Training Reconstruction Loss (MSE)')
ax.plot(range(1, len(val_losses) + 1), val_losses, 'r--', label='Validation Reconstruction Loss (MSE)')
ax.set_xlabel('Epoch', fontweight='bold')
ax.set_ylabel('Masked Reconstruction Loss (MSE)', fontweight='bold')
ax.set_title('LIMU-BERT Self-Supervised Pretraining Learning Curve', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
curve_path = plots_dir / 'limu_bert_training_curve.png'
plt.savefig(curve_path, dpi=200)
plt.close()
print(f'Learning curve saved: {curve_path}')

# Save training summary
summary = {
    'model': 'LIMU-BERT',
    'epochs_trained': len(train_losses),
    'best_val_loss': round(float(best_val_loss), 6),
    'final_train_loss': round(float(train_losses[-1]), 6),
    'trainable_params': param_count,
    'checkpoint': str(best_ckpt_path.relative_to(PROJECT_ROOT))
}
with open(PROJECT_ROOT / 'results' / 'limu_bert_results.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('LIMU-BERT pretraining results saved to results/limu_bert_results.json.')